# 05 - Test top-5 per model (`test_with_result.csv`)

Copies `test.csv` to `test_with_result.csv` and adds 5 columns, one per model, each holding that model's **top-5 predicted `hotel_cluster` ids** (space separated, best first) for every test row.

Protocol: **autoregressive** (primary). Inside the test window the true cluster of a booking never enters a later history of the same user; the model's own top-1 prediction for the earlier booking is used instead. Earlier-window bookings keep their true cluster. Reference seed 2022, hyper-parameters read from `results/week6_retrain/*.json` (chosen on valid in notebooks 01-03).

Models: plain, plain + query embedding, plain + query embedding + sameDest (main), ItemKNN, logistic regression. Trained SMLP4Rec weights are loaded from the `best.pth` of the run folders; ItemKNN and logistic regression are refit (not stored on disk). Kernel: `smlp4rec`.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch  # torch before pandas (Windows c10.dll)
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.query_features import build_vocab, encode
from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.autoregressive import score_view
from nbp.eval.metrics import ndcg_at_k, target_rank
from nbp.models import scorers as S
from nbp.paths import CONFIGS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
SEED = SPLIT["seeds"][0]
TOPN = 5
BATCH = int(TRAIN_CFG["eval_batch"])
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
RES = ROOT / "results" / "week6_retrain"
CSV = PROCESSED / "expedia_date_split" / "csv"


def res(name):
    path = RES / ("itemknn.json" if name == "itemknn" else f"{name}_s{SEED}.json")
    return json.loads(path.read_text(encoding="utf-8"))


def run_dir(p):  # the json stores an absolute path; keep the folder name only
    return ROOT / "experiments" / Path(str(p).replace("\\", "/")).name


rows = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows["codes"]
N_ITEMS = int(rows["y"].max()) + 1
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
v_test = views["test"]
print("test rows", len(v_test), "| items", N_ITEMS)

TOP = {}   # model key -> (n_test, TOPN) hotel_cluster ids, row order of the test view
RANK = {}  # model key -> rank of the true cluster (checked against the stored ranks)


def run_ar(key, score_fn):
    t0 = time.time()
    sc = score_view(v_test, score_fn, autoregressive=True, batch=BATCH)
    TOP[key] = R.top_k(sc.scores, TOPN).astype(np.int64) - 1  # model id -> hotel_cluster
    assert (TOP[key] >= 0).all()
    RANK[key] = target_rank(sc.scores, Y["test"])
    print(f"{key}: AR test NDCG@{K_MON} {ndcg_at_k(RANK[key], K_MON):.4f} ({time.time() - t0:.0f}s)")

test rows 378788 | items 101


## 1. Plain SMLP4Rec (history only)

In [2]:
meta = res("plain")
cfg = S.model_config(MODEL_CFG, {"seed": SEED, "hidden_dropout_prob": meta["hidden_dropout_prob"]})
plain = S.build_plain(cfg, N_ITEMS)
plain.load_state_dict(torch.load(run_dir(meta["run_dir"]) / "best.pth", map_location="cpu", weights_only=False))
plain.eval()
run_ar("plain", S.plain_scorer(plain))

plain: AR test NDCG@10 0.1843 (14s)


## 2. Plain + query embedding, and 3. + sameDest (main model)

In [3]:
meta_q, meta_m = res("query"), res("main")
train_rows = np.flatnonzero(rows["window"] == TRAIN)
vocab = build_vocab(codes, rows["event"][train_rows])  # train window only, as in notebook 02
Q = torch.from_numpy(encode(codes, vocab))
field_sizes = [len(v) + 1 for v in vocab]
cfg = S.model_config(MODEL_CFG, {"seed": SEED, "hidden_dropout_prob": meta_q["hidden_dropout_prob"]})
query = S.build_query(cfg, N_ITEMS, Q, field_sizes)
query.load_state_dict(torch.load(run_dir(meta_q["run_dir"]) / "best.pth", map_location="cpu", weights_only=False))
query.eval()
SD = meta_m["samedest"]
print("sameDest", {k: SD[k] for k in ("alpha", "w", "recency")})
q_sf = S.query_scorer(query)
run_ar("query", q_sf)
run_ar("main", S.samedest_scorer(q_sf, SD["alpha"], SD["w"], SD["recency"]))

sameDest {'alpha': 0.1, 'w': 0.5, 'recency': 0.7}


query: AR test NDCG@10 0.4446 (15s)


main: AR test NDCG@10 0.4519 (23s)


## 4-5. Baselines: ItemKNN and logistic regression (destination + trip context, no history)

In [4]:
import subprocess

# ItemKNN needs `implicit`, installed only in the project .venv: the baselines are refit and scored there
# (scripts/export_baseline_top5.py, same code as notebook 03) and read back here.
VENV_PY = ROOT / ".venv" / "Scripts" / "python.exe"
subprocess.run([str(VENV_PY), str(ROOT / "scripts" / "export_baseline_top5.py")], check=True, cwd=ROOT)
z = np.load(INTERIM / "week6" / f"top5_baselines_s{SEED}.npz")
assert np.array_equal(z["event"], rows["event"][idx["test"]])
for key in ("itemknn", "logreg"):
    TOP[key] = z[key].astype(np.int64)
    RANK[key] = z[f"rank_{key}"]

## Checks against the stored results

Each model's AR test ranks recomputed here must equal the ranks stored by notebooks 01-03 (`data/interim/week6/ranks_*_s2022.npz`).

In [5]:
for key in TOP:
    name = f"ranks_{key}" + ("" if key == "itemknn" else f"_s{SEED}") + ".npz"
    z = np.load(INTERIM / "week6" / name)
    same = float((z["test_ar"] == RANK[key]).mean())
    print(f"{key}: share of rows with the stored AR rank = {same:.6f}")
    assert same > 0.999, f"{key} differs from the stored ranks"

plain: share of rows with the stored AR rank = 1.000000
query: share of rows with the stored AR rank = 1.000000
main: share of rows with the stored AR rank = 1.000000
itemknn: share of rows with the stored AR rank = 1.000000
logreg: share of rows with the stored AR rank = 1.000000


## Write `test_with_result.csv`

In [6]:
src = pd.read_csv(CSV / "test.csv")
ev_test = rows["event"][idx["test"]]
assert np.array_equal(src["event_id"].to_numpy(), ev_test), "test.csv row order differs from the test view"
assert (src["hotel_cluster"].to_numpy() + 1 == Y["test"]).all()
old = np.array([[int(x) for x in s.split()[:TOPN]] for s in src["top10_main_ar"]])
print("main top-5 equals first 5 of top10_main_ar:", float((old == TOP["main"]).all(1).mean()))

out = src.copy()  # every column of test.csv is kept
NAMES = {"plain": "top5_plain", "query": "top5_plain_query", "main": "top5_main",
         "itemknn": "top5_itemknn", "logreg": "top5_logreg"}
for key, col in NAMES.items():
    out[col] = [" ".join(map(str, t)) for t in TOP[key]]
out.to_csv(CSV / "test_with_result.csv", index=False)
print(out.shape, "->", CSV / "test_with_result.csv")
out[["hotel_cluster", *NAMES.values()]].head()

main top-5 equals first 5 of top10_main_ar: 1.0


(378788, 46) -> C:\workspace\PROJECTS\next-best-product-recommendation\data\processed\expedia_date_split\csv\test_with_result.csv


,hotel_cluster,top5_plain,top5_plain_query,top5_main,top5_itemknn,top5_logreg
0,82,82 59 30 43 29,82 62 78 67 90,82 62 78 67 90,82 28 99 95 30,82 62 90 85 78
1,86,46 70 98 64 41,56 70 98 69 41,70 98 56 69 41,70 41 98 46 48,56 70 98 41 68
2,57,39 32 77 50 47,12 85 57 36 62,12 85 57 36 62,32 77 39 17 50,62 85 36 12 82
3,89,91 59 48 28 18,62 5 61 82 85,62 5 61 82 85,91 48 42 59 28,29 68 82 62 30
4,18,91 59 48 28 18,77 32 21 13 28,77 32 21 13 28,91 48 42 59 28,77 13 21 16 42


In [7]:
# recall@5 recomputed from the written columns (true cluster among the 5 listed ids)
chk = pd.read_csv(CSV / "test_with_result.csv")
for col in NAMES.values():
    hit = np.array([str(h) in s.split() for h, s in zip(chk["hotel_cluster"], chk[col])])
    print(f"{col}: test AR recall@5 = {hit.mean():.4f}")

top5_plain: test AR recall@5 = 0.2187
top5_plain_query: test AR recall@5 = 0.5546


top5_main: test AR recall@5 = 0.5606
top5_itemknn: test AR recall@5 = 0.2169


top5_logreg: test AR recall@5 = 0.4832
